In this notebook, we're going to be fitting our first model for Emotion/Stress detection, which is an SVM Classifier Model.

In [1]:
import numpy as np 
import pandas as pd
from sklearn import svm 
from sklearn.preprocessing import StandardScaler

Now, we load our processed data and store it in the following containers, ready to fit in the data. 

In [2]:
train_df = pd.read_csv("/Users/almarai/Private Projects/CogniScan/CogniScan/data/processed/train_data_processed.csv")
test_df = pd.read_csv("/Users/almarai/Private Projects/CogniScan/CogniScan/data/processed/test_data_processed.csv")

In [3]:
train_df.head()

,0,1,2,3,4,5,6,7,8,9,...,2295,2296,2297,2298,2299,2300,2301,2302,2303,emotion
0,70,80,82,72,58,58,60,63,54,58,...,182,183,136,106,116,95,106,109,82,0
1,151,150,147,155,148,133,111,140,170,174,...,108,95,108,102,67,171,193,183,184,0
2,231,212,156,164,174,138,161,173,182,200,...,138,152,122,114,101,97,88,110,152,0
3,24,32,36,30,32,23,19,20,30,41,...,126,132,132,133,136,139,142,143,142,0
4,4,0,0,0,0,0,0,0,0,0,...,34,31,31,31,27,31,30,29,30,2


In [4]:
X_train = train_df.drop(columns = ["emotion"])
y_train = train_df["emotion"]

X_test = test_df.drop(columns = ["emotion"])
y_test = test_df["emotion"]

# #Before fitting out model, we need to scale our features and target
# scaler = StandardScaler()
# scaler.fit(X_train)

# X_train_scaled = scaler.transform(X_train)
# X_test_scaled = scaler.transform(X_test)

In [5]:
X_train_small = train_df.iloc[:5000,:].drop(columns = ["emotion"])
y_train_small = train_df.iloc[:5000,:]["emotion"]
X_train_small.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Columns: 2304 entries, 0 to 2303
dtypes: int64(2304)
memory usage: 87.9 MB


In [6]:
scaler = StandardScaler()

scaler.fit(X_train_small)

X_train_scaled = scaler.transform(X_train_small)

Now we create a basic model, to check it's performance before we perform some hyperparameter tuning. 

In [7]:
# model = svm.SVC(
#     kernel='linear',
#     C=1.0,
#     gamma='scale',
#     probability=True
# )
# model.fit(X_train_scaled, y_train_small)

# y_pred = model.predict(X_train_scaled)

In [8]:
# from sklearn.metrics import confusion_matrix,classification_report

# print(confusion_matrix(y_pred, y_train_small))
# print(classification_report(y_pred,y_train_small))

So we've identified that the issue with the training of the model, is that the number of observations are too many and the computational time for training is becoming increasingly large. 

In [9]:
# import joblib

# joblib.dump(model, "/Users/almarai/Private Projects/CogniScan/CogniScan/models/svm_model.pkl")

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# === Load your dataset ===
# Replace these with your data loading logic
X = train_df.drop(columns = ["emotion"]).values  # Shape: (num_samples, height, width)
y = train_df["emotion"].values  # Labels: integers

# === Normalize pixel values ===
X = X / 255.0

# === Reshape to add channel dimension ===
X = X.reshape(-1, 48, 48, 1)  # Adjust if not 48x48

# === One-hot encode labels ===
num_classes = len(np.unique(y))
y = to_categorical(y, num_classes)

# === Split data ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === Build a simple CNN ===
model = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=(48, 48, 1)),
    MaxPooling2D((2,2)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# === Train ===
history = model.fit(
    X_train, y_train,
    epochs=10,               # Small number for CPU, adjust as needed
    batch_size=32,
    validation_data=(X_test, y_test)
)

# === Evaluate ===
loss, acc = model.evaluate(X_test, y_test, verbose=2)
print(f'Test accuracy: {acc:.4f}')

# === Save model ===
model.save("/Users/almarai/Private Projects/CogniScan/CogniScan/models/cnn_model.h5")

/opt/anaconda3/envs/cogniscan-env/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 18s 23ms/step - accuracy: 0.4534 - loss: 1.0306 - val_accuracy: 0.5355 - val_loss: 0.9502
Epoch 2/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 17s 23ms/step - accuracy: 0.5294 - loss: 0.9556 - val_accuracy: 0.5726 - val_loss: 0.8931
Epoch 3/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 17s 23ms/step - accuracy: 0.5718 - loss: 0.9014 - val_accuracy: 0.6080 - val_loss: 0.8559
Epoch 4/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - accuracy: 0.5974 - loss: 0.8680 - val_accuracy: 0.6200 - val_loss: 0.8271
Epoch 5/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - accuracy: 0.6209 - loss: 0.8282 - val_accuracy: 0.6346 - val_loss: 0.7977
Epoch 6/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 31s 43ms/step - accuracy: 0.6257 - loss: 0.8075 - val_accuracy: 0.6419 - val_loss: 0.7879
Epoch 7/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 38s 52ms/step - accuracy: 0.6398 - loss: 0.7849 - val_accuracy: 0.6616 - val_loss: 0.7735
Epoch 8/10
718/718 ━━━━━━━━━━━━━━━━━━━━ 34s 48ms/step - accuracy: 0.6567 - loss: 0.7588 - 

Test accuracy: 0.6691
